# LLM Skill Analysis

In [1]:
import json
import pandas as pd
import ollama
from pathlib import Path
import numpy as np

All csv files being written to 'analysis' folder

Uses ollama model 
1. pip install ollama
2. ollama run llama3


In [2]:
SUBSET_PATH = Path("llm_analysis_subset.csv")
BASELINES_PATH = Path("full_alexa_skills_with_baselines.csv")
DIRECTORY = Path("analysis")

MODEL = "llama3"

available = [m.model for m in ollama.list().models]
if any(MODEL in m for m in available):
    print("Using:", MODEL)
else:
    print(MODEL, "Not Found ERROR")

Using: llama3


## 1. Load Subset

In [3]:
# llm_analysis_subset.csv
subset = pd.read_csv(SUBSET_PATH)
print("Subset:", subset.shape)

# Alexa dataset
baselines = pd.read_csv(BASELINES_PATH)
print("Baselines:", baselines.shape)

subset[["Skill Name", "Invocation Name", "Category", "baseline_risk_score", "selection_reasons"]].head()

Subset: (1247, 62)
Baselines: (42874, 61)


,Skill Name,Invocation Name,Category,baseline_risk_score,selection_reasons
0,Cricket Facts,cricket facts,Business & Finance,0.834704,baseline_high_risk;high_genericity_ambiguity
1,bitcoin facts,crypto facts,Business & Finance,0.743449,baseline_high_risk;high_genericity_ambiguity
2,AI NVR Integration Test,my assistant,Business & Finance,0.660476,baseline_high_risk
3,Chore chart,chore chart,Business & Finance,0.656675,baseline_high_risk
4,Cryptocurrency,crypto currency,Business & Finance,0.639651,baseline_high_risk


## 2. Prompt 

In [4]:
def build_prompt(row):
    return f"""
You are analyzing potential security and usability risks of Alexa skill invocation names.

Evaluate the following skill and return a structured JSON response.

Skill Information:
- Skill Name: {row['Skill Name']}
- Invocation Name: {row['Invocation Name']}
- Category: {row['Category']}
- Description: {row['Description']}
- Average Rating: {row['Average Rating']}
- Number of Ratings: {row['Number of Rating']}

Context:
- Invocation names are spoken by users
- Users may mispronounce, abbreviate, or paraphrase.
- Attackers may create similar-sounding or similar-looking names.

Evaluate the following dimensions (0-5 scale):
1. Genericity Risk (how generic/common the name is)
2. Ambiguity Risk (multiple plausible meanings or interpretations)
3. Semantic Confusion Risk (could be confused with other skills)
4. Squatting Risk (ease of imitation or malicious variants)
5. Overall Risk (holistic judgment)

Return ONLY valid JSON in this format:

{{
    "genericity_risk": int,
    "ambiguity_risk": int,
    "semantic_confusion_risk": int,
    "squatting_risk": int,
    "overall_risk": int,
    "reasoning": "concise explanation (2-4 sentences)"
}}
"""

## 3. Load Functions

In [5]:
SCORE_FIELDS = ["genericity_risk", "ambiguity_risk", "semantic_confusion_risk",
                "squatting_risk", "overall_risk"]

def run_model(prompt):
    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    return resp["message"]["content"].strip()


def parse_response(resp):
    base = {}

    for field in SCORE_FIELDS:
        base["llm_" + field] = None


    base["llm_reasoning"] = None
    if resp is None:
        return base

    start = resp.find("{")
    end = resp.rfind("}")

    if start == -1 or end == -1:
        return base

    try:
        json_str = resp[start:end + 1].replace("\\'", "'")
        obj = json.loads(json_str)


        for field in SCORE_FIELDS:
            if field in obj:
                base["llm_" + field] = int(obj[field])
        base["llm_reasoning"] = str(obj.get("reasoning", ""))
    except (json.JSONDecodeError, KeyError, ValueError):
        pass

    return base

## 4. Run Analysis

In [6]:
new_rows = []

for index, row in subset.iterrows():
    raw_response = run_model(build_prompt(row))
    parsed_data  = parse_response(raw_response)

    combined_row = row.to_dict()
    combined_row.update(parsed_data)
    new_rows.append(combined_row)

    if (index + 1) % 25 == 0:
        print(index + 1, "rows completed")

results_df = pd.DataFrame(new_rows)
print("Results:", results_df.shape)

25 rows completed
50 rows completed
75 rows completed
100 rows completed
125 rows completed
150 rows completed
175 rows completed
200 rows completed
225 rows completed
250 rows completed
275 rows completed
300 rows completed
325 rows completed
350 rows completed
375 rows completed
400 rows completed
425 rows completed
450 rows completed
475 rows completed
500 rows completed
525 rows completed
550 rows completed
575 rows completed
600 rows completed
625 rows completed
650 rows completed
675 rows completed
700 rows completed
725 rows completed
750 rows completed
775 rows completed
800 rows completed
825 rows completed
850 rows completed
875 rows completed
900 rows completed
925 rows completed
950 rows completed
975 rows completed
1000 rows completed
1025 rows completed
1050 rows completed
1075 rows completed
1100 rows completed
1125 rows completed
1150 rows completed
1175 rows completed
1200 rows completed
1225 rows completed
Results: (1247, 68)


In [12]:
results_df.to_csv(DIRECTORY/"llm_results.csv")
print("saved")

saved


## 5. Score Distributions

In [7]:
llm_columns = []

for field in SCORE_FIELDS:
    column_name = "llm_"+str(field) 
    llm_columns.append(column_name)  

new_data = results_df.dropna(subset=llm_columns).copy()
description = new_data[llm_columns].describe().round(2)

## 6. Compare LLM vs Baseline

In [8]:
new_data["llm_composite"] = new_data["llm_overall_risk"]

corr = new_data[["baseline_risk_score", "llm_composite"]].corr().iloc[0, 1]
print(f"baseline_risk_score vs llm_overall_risk: {corr:.5f}")

pop_quartiles = baselines["baseline_risk_score"].quantile([0.25, 0.5, 0.75])
new_data["baseline_population_tier"] = pd.cut( new_data["baseline_risk_score"],
    bins=[-np.inf, pop_quartiles[0.25], pop_quartiles[0.5], pop_quartiles[0.75], np.inf],
    labels=["Q1 low", "Q2", "Q3", "Q4 high"],
)

new_data["llm_quartile"] = pd.qcut(new_data["llm_composite"], q=4, labels=False, duplicates="drop")

print()
print("LLM score distribution:")
print(new_data["llm_composite"].value_counts().sort_index())


print()
print("Baseline vs LLM overall risk:")
print(pd.crosstab(new_data["baseline_population_tier"], new_data["llm_composite"]))

comparison_cols = [
    "Skill Name", "Invocation Name", "Category",
    "baseline_risk_score", "baseline_population_tier",
    "llm_genericity_risk", "llm_ambiguity_risk",
    "llm_semantic_confusion_risk", "llm_squatting_risk",
    "llm_overall_risk", "llm_reasoning",
]
new_data[comparison_cols].to_csv(DIRECTORY / "llm_vs_baseline.csv", index=False)
print()
print("Wrote")

baseline_risk_score vs llm_overall_risk: 0.23038

LLM score distribution:
llm_composite
3    868
4    379
Name: count, dtype: int64

Baseline vs LLM overall risk:
llm_composite               3    4
baseline_population_tier          
Q1 low                    213   67
Q2                        167   35
Q3                        135   53
Q4 high                   353  224

Wrote


## 7. False positives vs False Negatives

In [9]:
fp = new_data[
    (new_data["baseline_risk_score"] >= new_data["baseline_risk_score"].quantile(0.75)) &
    (new_data["llm_composite"] <=new_data["llm_composite"].quantile(0.25))
].sort_values("baseline_risk_score", ascending=False)

print(f"False Positives: {len(fp)} skills")
fp[["Skill Name", "Invocation Name", "Category",
    "baseline_risk_score", "llm_composite", "llm_reasoning"]].head(10)

False Positives: 161 skills


,Skill Name,Invocation Name,Category,baseline_risk_score,llm_composite,llm_reasoning
611,cricket,cricket facts,Movie and TV,0.834704,3,The invocation name 'cricket facts' is moderat...
457,Science Facts,science facts,Kids,0.819611,3,The invocation name 'science facts' is somewha...
190,science info,science facts,Education & Reference,0.819611,3,The invocation name 'science facts' is somewha...
296,science facts,science facts,Game and Trivia,0.819611,3,The invocation name 'science facts' is moderat...
297,Sci Facts,science facts,Game and Trivia,0.819611,3,The invocation name 'science facts' is somewha...
191,Facto,science facts,Education & Reference,0.819611,3,The invocation name 'science facts' is somewha...
461,Food Fact,food facts,Kids,0.816546,3,The invocation name 'food facts' is somewhat g...
242,Food Facts,food facts,Food and Drink,0.816546,3,The invocation name 'food facts' is somewhat g...
241,Food Facts,food facts,Food and Drink,0.816546,3,The invocation name 'food facts' is moderately...
303,Food Fact,food facts,Game and Trivia,0.816546,3,The invocation name 'food facts' is somewhat g...


In [13]:
fn = new_data[
    (new_data["baseline_risk_score"] <= new_data["baseline_risk_score"].quantile(0.25)) &
    (new_data["llm_composite"] >=new_data["llm_composite"].quantile(0.75))
].sort_values("llm_composite", ascending=False)

print(f"False Negatives: {len(fn)} skills")
fn[["Skill Name", "Invocation Name", "Category",
    "baseline_risk_score", "llm_composite", "llm_reasoning"]].head(10)

disagreement_cols = ["Skill Name", "Invocation Name", "Category",
                     "baseline_risk_score", "llm_composite", "llm_reasoning"]

fp[disagreement_cols].to_csv(DIRECTORY/"false_positives.csv", index=False)
fn[disagreement_cols].to_csv(DIRECTORY/"false_negatives.csv", index=False)

print()
print(f"Wrote analysis/false_positives.csv {len(fp)} rows")
print(f"Wrote analysis/false_negatives.csv {len(fn)} rows")

False Negatives: 71 skills

Wrote analysis/false_positives.csv 161 rows
Wrote analysis/false_negatives.csv 71 rows


## 8. Category Summary

In [11]:
summary = (
    
    new_data.groupby("Category").agg(
        skill_count = ("Skill Name","count"),
        mean_baseline = ("baseline_risk_score","mean"),
        mean_llm_genericity_risk = ("llm_genericity_risk","mean"),
        mean_llm_ambiguity_risk = ("llm_ambiguity_risk","mean"),
        mean_llm_semantic_confusion = ("llm_semantic_confusion_risk","mean"),
        mean_llm_squatting_risk = ("llm_squatting_risk","mean"),
        mean_llm_overall_risk = ("llm_overall_risk","mean"),
        mean_llm_composite = ("llm_composite","mean"),
    ).sort_values("mean_llm_overall_risk", ascending=False).reset_index()
)
summary.to_csv(DIRECTORY /"llm_category_summary.csv", index=False)
summary.round(2)

,Category,skill_count,mean_baseline,mean_llm_genericity_risk,mean_llm_ambiguity_risk,mean_llm_semantic_confusion,mean_llm_squatting_risk,mean_llm_overall_risk,mean_llm_composite
0,Health and Fitness,61,0.44,3.00,1.07,2.00,2.75,3.61,3.61
1,Sports,47,0.53,3.00,0.98,1.98,2.98,3.53,3.53
2,Education & Reference,56,0.58,3.11,0.93,2.04,3.14,3.48,3.48
3,Game and Trivia,56,0.57,3.07,1.02,2.02,3.25,3.45,3.45
4,Kids,52,0.56,3.08,0.96,2.04,3.04,3.40,3.40
5,Novelty and Humor,56,0.58,3.11,1.09,1.93,3.39,3.39,3.39
6,weather,52,0.43,3.00,1.00,2.00,3.19,3.38,3.38
7,productivity,49,0.51,3.12,1.04,2.00,3.45,3.33,3.33
8,News,46,0.46,3.09,0.96,1.96,3.37,3.33,3.33
9,Travel and Transportation,52,0.42,3.04,1.02,1.98,3.42,3.31,3.31
